In [1]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals. 

In [6]:
import numpy as np 
import qiskit_metal as metal
from qiskit_metal import draw, Dict 
from qiskit_metal.qlibrary.core import QComponent 
#from global_functions import rotate_point

In [7]:
%reload_ext autoreload 
%autoreload 2 

In [8]:
# Function that rotates points by a certain degree. 
# This will be used to keep component nodes sticking to the same point 
# Even while undergoing a rotation from defining an `orientation` option. 

def rotate_point(point, angle):
    """
    Rotates a 2D point about the origin.
    
    Args:
        point (array-like): The (x, y) coordinates
        angle (float): The rotation angle in radians
    
    Returns:
        np.array: rotated point
        
    """
    
    rot_matrix = np.array([[np.cos(angle), -np.sin(angle)],
                           [np.sin(angle),  np.cos(angle)]])
    return np.dot(rot_matrix, point)

In [9]:
design = metal.designs.DesignPlanar() 
gui = metal.MetalGUI(design) 
design.overwrite_enabled = True 

01:50PM 34s CRITICAL [_qt_message_handler]: line: 0, func: None(), file: None  WARNING: Populating font family aliases took 181 ms. Replace uses of missing font family "Courier" with one that exists to avoid this cost. 



In [5]:
%metal_heading Junction Components

In [6]:
%metal_print Bandage

In [21]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals. 

""" Bandage. 

.. code-block:: 

                        `top` 
      `top_left` _________|_________ `top_right` 
                |                   |
                |                   |
                |                   | 
                |                   | 
       `left` - |         *         | - `right`
                |                   |
                |                   |  
                |                   | 
                |                   | 
                |                   | 
   `bottom_left` ‾‾‾‾‾‾‾‾‾|‾‾‾‾‾‾‾‾‾ `bottom_right` 
                      `bottom` 

 
""" 

import numpy as np 
from qiskit_metal import draw, Dict 
from qiskit_metal.qlibrary.core import QComponent 

class Bandage(QComponent): 
    """ 
    Inherits `QComponent` class. 

    Description: 
        A rectangular bandage structure. 

    Options: 
        * width   : width of the rectangle 
        * height  : height of the rectangle 
        * pos_x/_y: position of the bandage on chip
        * orientation : 0-> is parallel to x-axis, with orientation (in degrees) counterclockwise 
        * layer   : the layer number for the bandage

    """ 

    # Default drawing options 
    default_options = Dict(
        width='100um', 
        height='200um',
        pos_x='0um', 
        pos_y='0um', 
        orientation='0', 
        layer='1', 
    )

    # Component metadata
    # Name prefix for component, if user doesn't provide name 
    component_metadata = Dict( 
        short_name='bandage', 
        _qgeometry_table_poly='True', 
    )

    def make(self): 
        """ Convert self.options into QGeometry. """ 
        p = self.parse_options() # string options -> numbers 
        
        bandage = draw.rectangle(p.width, p.height)
        bandage = draw.rotate(bandage, p.orientation)
        bandage = draw.translate(bandage, p.pos_x, p.pos_y)

        # Converting drawing to qgeometry 
        self.add_qgeometry('poly', {'bandage': bandage}, layer=p.layer)

        # Positioning nodes 
        nodes = Dict() 
        nodes.origin = np.zeros(2)
        nodes.top = np.array((0, +p.height/2)) 
        nodes.bottom = np.array((0, -p.height/2))
        nodes.left = np.array((-p.width/2, 0)) 
        nodes.right = np.array((+p.width/2, 0))
        nodes.top_left = nodes.top + nodes.left 
        nodes.top_right = nodes.top + nodes.right 
        nodes.bottom_left = nodes.bottom + nodes.left 
        nodes.bottom_right = nodes.bottom + nodes.right

        # Moving all nodes along with component
        translation_vec = np.array((p.pos_x, p.pos_y)) 
        theta = np.deg2rad(p.orientation)
        for key, point in nodes.items():
            rotated = rotate_point(point, theta)
            nodes[key] = rotated + translation_vec
            
        self.nodes = nodes 
        return 

    def node(self, key): 
        return self.nodes.get(key, None) 

In [4]:
%metal_print BridgeFreeJunction

In [76]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals. 

""" BridgeFreeJunction. 

.. code-block:: 

                      `+y` 
               ________|_____
             /        | |     ∖ 
            |      ___|_|_*    | 
            |    |        |____|              
            |      `+x_+y`|____|- `+x`              
            |    |        |    |              
            |    |        |    | 
            |     ‾‾‾‾‾‾‾‾     | 
             ∖                / 
               ‾‾‾‾‾‾‾‾‾‾‾‾‾‾

"""
             

class BridgeFreeJunction(QComponent): 
    """ 
    Inherits `QComponent` Class. 

    Description: 
        A bridge-free junction design similar to the manhattan style junctions. 

    Options: 
        * width       : The width of the junction overlap rectangle 
        * height      : The height of the junction overlap rectangle 
        * wire_width  : The width of the junction wires 
        * wire_x_offset: The offset in x direction w.r.t origin -- sign doesn't matter
        * wire_y_offset: The offset in y direction w.r.t origin -- sign doesn't matter 
        * pos_x/_y    : position of junction on chip 
        * fillet      : The fillet radius of the corners of the junction overlap rectangle 
        * undercut    : The size of the undercut region around the junction overlap rectangle 
        * orientation : 0-> is parallel to x-axis, with orientation (in degrees) counterclockwise 
        * origin: A string representing the corner of the rectangle the wires will be placed next to
                  Valid Options: 
                      * 'upper right' DEFAULT 
                      * 'upper left' 
                      * 'lower right' 
                      * 'lower left' 
        * layer: the layer number for the junction 
        * undercut_layer: the layer number for the undercut 


    """ 

    # Default drawing options 
    default_options = Dict(
        width='100um',
        height='100um', 
        wire_width='5um', 
        wire_x_offset='5um', 
        wire_y_offset='5um', 
        pos_x='0um', 
        pos_y='0um', 
        fillet='0um', 
        undercut='25um', 
        orientation='0', 
        origin='upper right',
        layer='1',
        undercut_layer='2', 
    ) 

    # Component metadata
    # Name prefix for component, if user doesn't provide name 
    component_metadata = Dict( 
        short_name='bridgefreejunction', 
        _qgeometry_table_poly='True', 
    )

    def make(self): 
        """ Convert self.options into QGeometry. """ 
        p = self.parse_options() # string options -> numbers 
        nodes = Dict() 

        # junction overlap rectangle 
        junc_rect = draw.rectangle(p.width, p.height) 

        # surrounding undercut 
        undercut = draw.rectangle(p.width+2*p.undercut, p.height+2*p.undercut) 
        undercut = undercut.buffer(p.fillet) 

        if p.origin == 'upper right': 
            nodes.origin = np.array((+p.width/2, +p.height/2)) 
            p.wire_x_offset = -abs(p.wire_x_offset) 
            p.wire_y_offset = -abs(p.wire_y_offset)
            x_translation = nodes.origin + [p.wire_x_offset, p.undercut/2+p.fillet/2] 
            y_translation = nodes.origin + [p.undercut/2+p.fillet/2, p.wire_y_offset] 
        elif p.origin == 'upper left': 
            nodes.origin = np.array((-p.width/2, +p.height/2)) 
            p.wire_x_offset = +abs(p.wire_x_offset) 
            p.wire_y_offset = -abs(p.wire_y_offset) 
            x_translation = nodes.origin + [p.wire_x_offset, p.undercut/2+p.fillet/2] 
            y_translation = nodes.origin + [-p.undercut/2-p.fillet/2, p.wire_y_offset] 
        elif p.origin == 'lower right': 
            nodes.origin = np.array((+p.width/2, -p.height/2)) 
            p.wire_x_offset = -abs(p.wire_x_offset) 
            p.wire_y_offset = +abs(p.wire_y_offset) 
            x_translation = nodes.origin + [p.wire_x_offset, -p.undercut/2-p.fillet/2] 
            y_translation = nodes.origin + [p.undercut/2+p.fillet/2, p.wire_y_offset] 
        elif p.origin == 'lower left': 
            nodes.origin = np.array((-p.width/2, -p.height/2)) 
            p.wire_x_offset = +abs(p.wire_x_offset) 
            p.wire_y_offset = +abs(p.wire_y_offset) 
            x_translation = nodes.origin + [p.wire_x_offset, -p.undercut/2-p.fillet/2] 
            y_translation = nodes.origin + [-p.undercut/2-p.fillet/2, p.wire_y_offset]
        else: 
            raise Exception('`origin` option not defined correctly') 

        # Wires  
        # `wire_x` is the wire that gets offset translated along x-direction 
        # `wire_y` is the wire that gets offset translated along y-direction
        wire_x = draw.rectangle(p.wire_width, p.undercut+p.fillet) 
        wire_y = draw.rectangle(p.undercut+p.fillet, p.wire_width) 
        
        wire_x = draw.translate(wire_x, *x_translation)
        wire_y = draw.translate(wire_y, *y_translation)  

        junction = draw.union([junc_rect, wire_x, wire_y]) 
        undercut = draw.subtract(undercut, junction) 

        # Translations & Rotations 
        geom_list = [junction, undercut] 
        geom_list = draw.rotate(geom_list, p.orientation, origin=(0, 0)) 
        geom_list = draw.translate(geom_list, p.pos_x, p.pos_y) 
        [junction, undercut] = geom_list
        
        # Converting drawing to qgeometry  
        self.add_qgeometry('poly', {'junction': junction}, layer=p.layer) 
        self.add_qgeometry('poly', {'undercut': undercut}, layer=p.undercut_layer) 

        # Positioning nodes
        nodes['+x_+y'] = nodes.origin + [p.wire_x_offset, p.wire_y_offset] 
        if p.origin == 'upper right':
            nodes['+x'] = nodes.origin + [p.undercut, p.wire_y_offset] 
            nodes['+y'] = nodes.origin + [p.wire_x_offset, p.undercut]
        if p.origin == 'upper left':
            nodes['+x'] = nodes.origin + [-p.undercut, p.wire_y_offset] 
            nodes['+y'] = nodes.origin + [p.wire_x_offset, p.undercut]
        if p.origin == 'lower right':
            nodes['+x'] = nodes.origin + [p.undercut, p.wire_y_offset] 
            nodes['+y'] = nodes.origin + [p.wire_x_offset, -p.undercut]
        if p.origin == 'lower left':
            nodes['+x'] = nodes.origin + [-p.undercut, p.wire_y_offset] 
            nodes['+y'] = nodes.origin + [p.wire_x_offset, -p.undercut]

        # Moving nodes along with component 
        translation_vec = np.array((p.pos_x, p.pos_y)) 
        theta = np.deg2rad(p.orientation) 
        for key, point in nodes.items(): 
            rotated = rotate_point(point, theta) 
            nodes[key] = rotated + translation_vec

        self.nodes = nodes 
        return 

    def node(self, key): 
        return self.nodes.get(key, None) 

In [15]:
%metal_print JunctionLead

In [80]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals.  

""" BridgeFreeJunction. 

.. code-block:: 


                                           `lead_1
            |‾‾‾‾‾‾‾‾‾‾‾‾|‾‾‾ --- ___          | 
            |            |            ‾‾‾|‾‾‾‾‾|‾‾∖
           *|            |               |  /‾‾|   |- `lead_0`
            |            |               | /   |  /
            |            |    ___ --- ‾‾‾ /‾‾‾‾|‾‾
             ‾‾‾‾‾‾‾‾‾‾‾‾ ‾‾‾            /  `lead_2` 
                                        /     
                                  `inner_lead`
"""


class JunctionLead(QComponent): 
    """ 
    Inherits `QComponent` Class. 

    Description: 
        Large Junction leads. 
        These are the large junction leads that connect the SQUID loop or junction wires to 
        the capacitor pads.  

    Options: 
        * outer_length : The length of the wide section.
                         will be computed from the total length and the lengths of the other segments
        * outer_width  : The width of the wide section 
        * inner_length : The length of the narrow section 
        * inner_width  : The width of the narrow section 
        * taper_length : The length of the taper 
        * pos_x/_y     : `origin` position of junction lead on chip 
                         `origin` is the left-edge-center of the wide-section 
        * fillet : The radius of the fillet. Only the two corners at the end of the narrow section 
                   will be filleted. 
        * extension    : The length of the narrow section past the junction wires. 
        * orientation  : 0-> is parallel to x-axis, with orientation (in degrees) counterclockwise 
        * layer: the layer number for the JunctionLead


    """ 

    # Default drawing options 
    default_options = Dict( 
        outer_length='500um', 
        outer_width='250um', 
        inner_length='250um', 
        inner_width='125um', 
        taper_length='700um', 
        fillet='0um', 
        extension='0um', 
    )

    # Component metadata 
    # Name prefix for component, if user doesn't provide name 
    component_metadata = Dict( 
        short_name='JunctionLead', 
        _qgeometry_table_poly='True', 
    )

    def make(self): 
        """ Convert self.options into QGeometry. """ 
        p = self.parse_options() 
        nodes = Dict() 

        # Wide section 
        wide_rect = draw.rectangle(p.outer_length, p.outer_width) 

        # Tapered section -- Built via subtract
        tapered_rect = draw.rectangle(p.taper_length, p.outer_width) 

        upper_triangle = draw.Polygon([
            (-p.taper_length/2, +p.outer_width/2), 
            (+p.taper_length/2, +p.outer_width/2), 
            (+p.taper_length/2, +p.inner_width/2), 
        ]) 
        lower_triangle = draw.Polygon([
            (-p.taper_length/2, -p.outer_width/2), 
            (+p.taper_length/2, -p.outer_width/2), 
            (+p.taper_length/2, -p.inner_width/2), 
        ]) 
        
        tapered_rect = draw.subtract(tapered_rect, upper_triangle) 
        tapered_rect = draw.subtract(tapered_rect, lower_triangle) 
        
        # Inner section 
        inner_rect = draw.rectangle(p.inner_length, p.inner_width) 
        
        def fillet_right_corners(L, W, r, num_points=10): 
            """ 
            Returns a shapely polyon for a rectangle of (`L` x `W`) centered at 
            (0, 0), where only the two right corners are filleted with radius `r`. 

            This is for the filleted-narrow-end of the JunctionLead. 

            """ 

            if r > W/2 or r > L/2: 
                raise ValueError('Fillet radius is too large for the provided rectangle dimensions.') 

            # Non-filleted corners 
            top_left    = (-L/2, +W/2) 
            bottom_left = (-L/2, -W/2) 

            # Tangent points where the fillets begin/end 
            top_right_straight = (L/2-r, W/2) 
            bottom_right_straight = (L/2, -W/2) 

            # Top right fillet 
            center_tr = (L/2 - r, W/2 - r) 
            arc_tr = [] 
            for i in range(num_points + 1): 
                theta = np.pi/2 * (1 - i/num_points) 
                x = center_tr[0] + r * np.cos(theta) 
                y = center_tr[1] + r * np.sin(theta) 
                arc_tr.append((x, y))

            # Bottom right fillet 
            center_br = (L/2 - r, -W/2 + r) 
            arc_br = [] 
            for i in range(num_points + 1): 
                theta = 0 - (np.pi/2) * (i/num_points) 
                x = center_br[0] + r * np.cos(theta) 
                y = center_br[1] + r * np.sin(theta) 
                arc_br.append((x, y)) 

            coords = []
            coords.append(bottom_left)                
            coords.append(top_left)                    
            coords.append(top_right_straight)          
            coords.extend(arc_tr)                      
            coords.append((L/2, -W/2 + r))
            coords.extend(arc_br)                      
            coords.append(bottom_left)

            return draw.Polygon(coords)
            
        if p.extension != 0.0: 
            extended_rect = draw.rectangle(p.extension, p.inner_width)  
            extended_rect = fillet_right_corners(p.extension, p.inner_width, p.fillet) 
        else: 
            inner_rect = fillet_right_corners(p.inner_length, p.inner_width, p.fillet)
            extended_rect = draw.rectangle(0, 0) # default to null rectangle

        # Positioning all sections correctly 
        tapered_rect = draw.translate(tapered_rect, (p.outer_length + p.taper_length)/2, 0) 
        inner_rect = draw.translate(inner_rect, p.taper_length + (p.outer_length + p.inner_length)/2, 0)
        extended_rect = draw.translate(extended_rect, p.taper_length + p.inner_length + (p.outer_length + p.extension)/2, 0)
            
        # Translations & Rotations 
        junction_lead = draw.union([wide_rect, tapered_rect, inner_rect, extended_rect] )
        junction_lead = draw.rotate(junction_lead, p.orientation)
        junction_lead = draw.translate(junction_lead, p.pos_x + p.outer_length/2, p.pos_y) 

        # Converting drawing to qgeometry  
        self.add_qgeometry('poly', {'junction_lead': junction_lead}, layer = p.layer) 

        # Positioning nodes 
        nodes.origin = np.zeros(2)
        nodes.lead_0 = nodes.origin + [p.outer_length+p.taper_length+p.inner_length+p.extension, 0]
        nodes.lead_1 = nodes.lead_0 + [-p.extension, p.inner_width/2] 
        nodes.lead_2 = nodes.lead_0 + [-p.extension, -p.inner_width/2]
        nodes.inner_lead = nodes.lead_0 + [-p.extension, 0] 

        # Moving nodes along with component 
        # pivot necessary as origin is not defined at center of object
        pivot = np.array([(p.outer_length + p.taper_length + p.inner_length + p.extension)/2, 0])
        theta = np.deg2rad(p.orientation)
        translation_vec = np.array((p.pos_x, p.pos_y)) 
        for key, point in nodes.items(): 
            relative = point - pivot
            rotated_relative = rotate_point(relative, theta)
            rotated = pivot + rotated_relative
            nodes[key] = rotated + translation_vec

        self.nodes = nodes
        return  

    def node(self, key): 
        return self.nodes.get(key, None) 

In [ ]:
taper = draw.rectangle(15, 5, 0, 0) 

In [91]:
%metal_print JunctionArray

In [5]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals.  

""" BridgeFreeJunction. 

.. code-block::  

                     `wire1` 
        ________________|_______________
        ‾‾‾‾‾‾‾‾‾‾‾|‾‾‾|‾|‾‾‾‾‾‾‾‾‾‾‾‾‾‾
        ___________|___|_|______________
        ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾|‾|‾‾‾|‾‾‾‾‾‾‾‾‾‾
        _______________|_|___|__________
        ‾‾‾‾‾‾‾‾‾‾‾|‾‾‾|‾|‾‾‾‾‾‾‾‾‾‾‾‾‾‾
        ___________|___|_|______________
        ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾|‾|‾‾‾|‾‾‾‾‾‾‾‾‾‾
        _______________|*|___|__________
        ‾‾‾‾‾‾‾‾‾‾‾|‾‾‾|‾|‾‾‾‾‾‾‾‾‾‾‾‾‾‾
        ___________|___|_|______________
        ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾|‾|‾‾‾|‾‾‾‾‾‾‾‾‾‾
        _______________|_|___|__________
        ‾‾‾‾‾‾‾‾‾‾‾|‾‾‾|‾|‾‾‾‾‾‾‾‾‾‾‾‾‾‾
        ___________|___|_|______________
        ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾|‾|‾‾‾|‾‾‾‾‾‾‾‾‾‾
        _______________|_|___|__________
        ‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾|‾‾‾‾‾‾‾‾‾‾‾‾‾‾‾
                     `wire2`
        

""" 

class JunctionArray(QComponent): 
    """ 
    Inherits from `QComponent` Class.  

    Description: 
        A two-angle bridge free junction array. 
        This implements a junction array superinductor, primarily for fluxonium qubits. 
        Designed in the style of Masluk et al. 2020. 

    Options: 
        * n (int): Number of junction lines 
        * jx/jy  : Dimensions for junction lines 
        * wx/wy  : Dimensions for central vertical wire pads 
                   `wy` determines the gap between the junction lines 
        * undercut: length of the undercut leads on junction line edges 
                    AND alternating central pads  
        * orientation  : 0-> is parallel to x-axis, with orientation (in degrees) counterclockwise
        * pos_x/_y     : `origin` position of junction lead on chip 
                         `origin` is the left-edge-center of the wide-section 
        * layer    : The layer number for the junction lines 
        * undercut_layer   : The layer number for the edge undercut leads
        * alternating_layer: The layer number for the alternating undercut pads 
        

    """
    
    # Default drawing options 
    default_options = Dict( 
        n=20, 
        jx='100um', 
        jy='10um', 
        wx='20um', 
        wy='10um', 
        undercut='15um', 
        orientation='0', 
        pos_x='0um', 
        pos_y='0um', 
        layer='1', 
        undercut_layer='2', 
        alternating_layer='2' 
    )

    # Component metadata 
    # Name prefix for component, if user doesn't provide name 
    component_metadata = Dict( 
        short_name='JunctionArray', 
        _qgeometry_table_poly='True', 
    )

    def make(self): 
        """ Converts self.options into QGeometry. """ 
        p = self.parse_options() 
        nodes = Dict()
 
        spacing = p.jy + p.wy 
        positions = (np.arange(p.n) - (p.n-1)/2)*spacing 

        # Junction lines and edge undercut leads 
        junction_rects = []
        lead_rects     = []
        for y in positions: 
            junction_rect = draw.rectangle(p.jx, p.jy) 
            junction_rect = draw.translate(junction_rect, 0, y) 

            base_undercut = draw.rectangle(p.undercut, p.jy)
            undercuts = [ 
                draw.translate(base_undercut, -(p.undercut + p.jx)/2, y), 
                draw.translate(base_undercut, +(p.undercut + p.jx)/2, y),
            ] 

            junction_rects.append(junction_rect) 
            lead_rects.extend(undercuts)       

        # Combining all junction lines into one component 
        # Combining all edge undercut leads into one component 
        junction_rects = draw.union(*junction_rects) 
        lead_rects     = draw.union(*lead_rects) 


        wire_rects = []
        pad_rects  = []
        for i, y in enumerate(positions[:-1]):
            wire_rect = draw.rectangle(p.wx, p.wy)
            wire_rect = draw.translate(wire_rect, 0, y + (p.jy + p.wy)/2)

            undercut = draw.rectangle(p.undercut, p.wy)
            if i%2: 
                undercut = draw.translate(undercut, +(p.wx + p.undercut)/2, y + (p.jy + p.wy)/2) 
            else: 
                undercut = draw.translate(undercut, -(p.wx + p.undercut)/2, y + (p.jy + p.wy)/2)

            wire_rects.append(wire_rect) 
            pad_rects.append(undercut)

        wire_rects = draw.union(*wire_rects) 
        pad_rects  = draw.union(*pad_rects)

        # Translations & Rotations 
        geom_list = [junction_rects, wire_rects, lead_rects, pad_rects] 
        geom_list = draw.rotate(geom_list, p.orientation)
        geom_list = draw.translate(geom_list, p.pos_x, p.pos_y) 
        [junction_rects, wire_rects, lead_rects, pad_rects] = geom_list 

        # Converting drawings to qgeometry
        self.add_qgeometry('poly', {'junction_rects': junction_rects}, layer = p.layer)
        self.add_qgeometry('poly', {'wire_rects': wire_rects}, layer=p.layer) 
        self.add_qgeometry('poly', {'edge_leads': lead_rects}, layer=p.undercut_layer) 
        self.add_qgeometry('poly', {'inner_pads': pad_rects}, layer=p.alternating_layer) 

        # Positioning nodes 
        nodes.origin = np.zeros(2) 
        nodes.wire1 = np.array((0, max(positions) + p.jy/2)) 
        nodes.wire2 = np.array((0, min(positions) - p.jy/2)) 

        # Moving nodes along with component  
        translation_vec = np.array((p.pos_x, p.pos_y)) 
        theta = np.deg2rad(p.orientation) 

        for key, point in nodes.items():
            rotated = rotate_point(point, theta) 
            nodes[key] = rotated + translation_vec

        self.nodes = nodes 
        return 

    def node(self, key): 
        return self.nodes.get(key, None) 

In [ ]:
%metal_print ManhattanJunction 

In [14]:
%metal_print ManhattanJunction

In [28]:
# This code was written by Deval Deliwala.
# 
# This code is to be used by QNL. 
# 
# Any modifications or derivative works of this code must retain this 
# notice, and modified files need to carry a notice indicating 
# that they have been altered from the originals.   

""" ManhattanJunction 

.. code-block::   


                      `vlead0` 
                          | 
                          ‖ `inside_corner` 
                         |‾|   / 
                         | |  / 
          __________     | | / 
         |       _  |____|_|/___________ 
         |      | |=|    |*|            |=- `hlead_0` 
         |     / ‾ /|‾‾‾‾|‾|‾‾‾‾‾‾‾‾‾‾‾‾ 
          ‾‾‾‾/‾‾‾/‾     | | 
             /   /       | | `vlead1`  
        `hcontact`       | |  /
               /         | | / 
              /          | |/  
          `hlead1`    |‾‾ ‖ ‾‾| 
                      |  |‾|  | 
                      |  /‾   | 
                      | /     | 
                      |/      | 
                      /‾‾‾‾‾‾‾
                  `vcontact` 

""" 


class ManhattanJunction(QComponent): 
    """ 
    Inherits from `QComponent` Class. 

    Description: 
        QNL Manhattan Junctions. 
        This junction mask design has been optimized for wafer scale junction uniformity. 

    Options: 
        * junction_hw: Horizontal junction width 
        * junction_vw: Vertical junction width 
        * junction_hl: Horizontal junction length 
        * junction_vl: Vertical junction length 
        * wire_hw : Horizontal wire width 
        * wire_vw : Vertical wire width 
        * taper_length: Length of taper from wire width to junction width 
        * extra_hl: Extra length for horizontal junction segment AFTER intersection
        * extra_vl: Extra length for vertical junction segment   AFTER intersection
        * contact_nw : A multiplier specifying the width of the narrow contact piece 
                       This is multiplied by the junction width 
        * contact_ww : A multiplier specifying the width of the wide contact piece 
                       This is multiplied by the junction width 
        * contact_nl : The length of the narrow contact piece 
        * contact_wl : THe length of the wide contact piece 
        * undercut_l : The length of the undercut after the junction segments
        * undercut_w : The width of the undercut after the junction segments 
        * orientation  : 0-> is parallel to x-axis, with orientation (in degrees) counterclockwise
        * pos_x/_y     : `origin` position of junction lead on chip 
                         `origin` is the left-edge-center of the wide-section 
        * layer    : The layer number for the junction lines 
        * undercut_layer   : The layer number for the edge undercut leads 
        

    """

    # Default drawing options 
    default_options = Dict( 
        junction_hw='10um', 
        junction_vw='10um', 
        junction_hl='100um', 
        junction_vl='100um', 
        wire_hw='5um', 
        wire_vw='5um', 
        taper_length='20um', 
        extra_hl='10um', 
        extra_vl='50um', 
        contact_nw=0.5, 
        contact_ww=1, 
        contact_nl='2um', 
        contact_wl='5um', 
        undercut_l= '150um',
        undercut_w='30um', 
        orientation='0', 
        pos_x='0um', 
        pos_y='0um', 
        layer='1', 
        undercut_layer='2',
    ) 

    # Component metadata 
    # Name prefix for component, if user doesn't provide name 
    component_metadata = Dict( 
        short_name='ManhattanJunction', 
        _qgeometry_table_poly='True', 
    )

    def make(self): 
        """ Converts self.options into QGeometry. """ 
        p = self.parse_options() 
        nodes = Dict() 

        # Junction Rectangles 
        h_junction = draw.rectangle(p.junction_hl, p.junction_hw) 
        h_junction = draw.translate(h_junction, (p.junction_hl-p.junction_vw)/2, 0) 
        
        v_junction = draw.rectangle(p.junction_vl, p.junction_vw) 
        v_junction = draw.translate(v_junction, (p.junction_vl-p.junction_hw)/2, 0) 
        
        # Wire-Junction taper
        h_triangles = [ 
            draw.Polygon([
                (-p.taper_length/2, +p.junction_hw/2), 
                (+p.taper_length/2, +p.junction_hw/2), 
                (+p.taper_length/2, +p.wire_hw/2), 
            ]), 
            draw.Polygon([ 
                (-p.taper_length/2, -p.junction_hw/2), 
                (+p.taper_length/2, -p.junction_hw/2), 
                (+p.taper_length/2, -p.wire_hw/2), 
            ]), 
        ] 

        v_triangles = [ 
            draw.Polygon([
                (-p.taper_length/2, +p.junction_vw/2), 
                (+p.taper_length/2, +p.junction_vw/2), 
                (+p.taper_length/2, +p.wire_vw/2), 
            ]), 
            draw.Polygon([ 
                (-p.taper_length/2, -p.junction_vw/2), 
                (+p.taper_length/2, -p.junction_vw/2), 
                (+p.taper_length/2, -p.wire_vw/2), 
            ]), 
        ]

        h_taper = draw.rectangle(p.taper_length, p.junction_hw) 
        h_taper = draw.subtract(h_taper, h_triangles[0]) 
        h_taper = draw.subtract(h_taper, h_triangles[1]) 
        h_taper = draw.translate(h_taper, p.junction_hl + (-p.junction_vw+p.taper_length)/2, 0)

        v_taper = draw.rectangle(p.taper_length, p.junction_vw)
        v_taper = draw.subtract(v_taper, v_triangles[0]) 
        v_taper = draw.subtract(v_taper, v_triangles[1]) 
        v_taper = draw.translate(v_taper, p.junction_vl + (-p.junction_hw+p.taper_length)/2, 0)

        
        # Junction Extensions 
        h_extension = draw.rectangle(p.extra_hl, p.junction_hw) 
        h_extension = draw.translate(h_extension, -(p.junction_vw+p.extra_hl)/2, 0) 

        v_extension = draw.rectangle(p.extra_vl, p.junction_vw) 
        v_extension = draw.translate(v_extension, -(p.junction_hw+p.extra_vl)/2, 0) 
        
        # Narrow contacts 
        h_narrow = draw.rectangle(p.contact_nl, p.contact_nw*p.junction_hw)
        v_narrow = draw.rectangle(p.contact_nl, p.contact_nw*p.junction_vw) 

        h_narrow = draw.translate(h_narrow, -(p.extra_hl + (p.contact_nl + p.junction_vw)/2), 0) 
        v_narrow = draw.translate(v_narrow, -(p.extra_vl + (p.contact_nl + p.junction_hw)/2), 0)

        # Wide contacts  
        h_wide = draw.rectangle(p.contact_wl, p.contact_ww*p.junction_hw)
        v_wide = draw.rectangle(p.contact_wl, p.contact_ww*p.junction_vw) 

        h_wide = draw.translate(h_wide, -(p.extra_hl + p.contact_nl + (p.contact_wl + p.junction_vw)/2), 0) 
        v_wide = draw.translate(v_wide, -(p.extra_vl + p.contact_nl + (p.contact_wl + p.junction_hw)/2), 0)

        # Undercut pads 
        undercut = draw.rectangle(p.undercut_l, p.undercut_w) 
        h_undercut = draw.translate(undercut, -(p.extra_hl + (p.junction_vw+p.undercut_l)/2), 0)
        v_undercut = draw.translate(undercut, -(p.extra_vl + (p.junction_hw+p.undercut_l)/2), 0)

        # Combining all non-undercut components into `junction` 
        # Applying 90 deg rotation to vertical junction 
        h_junction = draw.union([h_junction, h_taper, h_extension, h_narrow, h_wide]) 
        v_junction = draw.union([v_junction, v_taper, v_extension, v_narrow, v_wide]) 

        v_undercut = draw.rotate(v_undercut, 90.0, origin=(0, 0))
        v_junction = draw.rotate(v_junction, 90.0, origin=(0, 0)) 
        junction = draw.union([h_junction, v_junction])  
        undercut = draw.union([h_undercut, v_undercut]) 

        # Translations & Rotations 
        geom_list = [junction, undercut] 
        geom_list = draw.rotate(geom_list, p.orientation, origin=(0, 0)) 
        geom_list = draw.translate(geom_list, p.pos_x, p.pos_y) 
        junction, undercut = geom_list

        # Converting drawing to qgeometry
        self.add_qgeometry('poly', {'junction': junction}, layer=p.layer)
        self.add_qgeometry('poly', {'undercut': undercut}, layer=p.undercut_layer) 

        # Positioning nodes 
        nodes.origin = np.zeros(2) 
        nodes.hlead0 = np.array((p.taper_length + p.junction_hl - p.junction_vw/2, 0)) 
        nodes.hlead1 = np.array((-(p.junction_vw/2 + p.extra_hl), 0)) 
        nodes.vlead0 = np.array((0, p.taper_length + p.junction_vl - p.junction_hw/2)) 
        nodes.vlead1 = np.array((0, -(p.junction_hw/2 + p.extra_vl)))
        nodes.hcontact = nodes.hlead1 + np.array((-p.contact_wl-p.contact_nl, 0))
        nodes.vcontact = nodes.vlead1 + np.array((0, -p.contact_wl-p.contact_nl))
        nodes.inside_corner = nodes.origin + np.array((p.junction_hw/2, p.junction_vw/2)) 

        # Moving nodes along with component  
        translation_vec = np.array((p.pos_x, p.pos_y)) 
        theta = np.deg2rad(p.orientation) 

        for key, point in nodes.items():
            rotated = rotate_point(point, theta) 
            nodes[key] = rotated + translation_vec

        self.nodes = nodes 
        return 

    def node(self, key): 
        return self.nodes.get(key, None) 